# Tutorial 2 — Data Discovery via Federated Analytics

This notebook explores all three registered datasets using the Rhino FCP's
federated metrics library. Every computation runs on the Rhino client —
**no raw data is ever returned to this notebook.**

**Can I do this in the UI instead?**
Yes — the FCP Dashboard provides a built-in Analytics view for each dataset.
Navigate to Dashboard → Projects → [Your Project] → Datasets → click a dataset
→ Analytics tab. The notebook gives you more flexibility and reproducibility.

---
**Prerequisites:**
- Tutorial 1 complete — paste your 6 UIDs in the Configuration cell

**Inputs:**
- `PATIENTS_DATASET_UID`, `ENCOUNTERS_DATASET_UID`, `PROCEDURES_DATASET_UID`

**Outputs / what you will learn:**
- Row counts confirming datasets are accessible
- Demographic distributions (gender, race, ethnicity) — exact values needed for Tutorial 4 mapping
- Data quality issues: casing inconsistencies, null rates, invalid values
- Visit type and procedure code distributions
- A Federated Dataset grouping (demonstrates multi-site pattern)
- Summary of issues to fix in Tutorial 3

## Configuration

In [ ]:
import rhino_health as rh
from rhino_health import ApiEnvironment
from rhino_health.lib.metrics import Count, Mean, StandardDeviation
from getpass import getpass

# From Tutorial 1 — paste your values here
PROJECT_UID            = "<YOUR-PROJECT-UID>"                   # REPLACE with your project UID
PATIENTS_DATASET_UID   = '<YOUR-PATIENTS-DATASET-UID>'          # REPLACE with your Patients dataset UID
ENCOUNTERS_DATASET_UID = '<YOUR-ENCOUNTERS-DATASET-UID>'        # REPLACE with your Encounters dataset UID
PROCEDURES_DATASET_UID = '<YOUR-PROCEDURES-DATASET-UID>'        # REPLACE with your Procedures dataset UID

if (PROJECT_UID == "" or PROJECT_UID == "<YOUR-PROJECT-UID>"):
    raise ValueError("Please fill in your project UID from Tutorial 1 before running this cell.")
if (PATIENTS_DATASET_UID == "" or ENCOUNTERS_DATASET_UID == "" or PROCEDURES_DATASET_UID == ""
    or PATIENTS_DATASET_UID == "<YOUR-PATIENTS-DATASET-UID>" or ENCOUNTERS_DATASET_UID == "<YOUR-ENCOUNTERS-DATASET-UID>" or PROCEDURES_DATASET_UID == "<YOUR-PROCEDURES-DATASET-UID>"):
    raise ValueError("Please fill in the dataset UIDs from Tutorial 1 before running this cell.")
print(f"System configured successfully.")

## Step 1 — Authenticate

In [ ]:
my_username = "<YOUR-USERNAME>"  # REPLACE
session = rh.login(username=my_username, password=getpass(), rhino_api_url=ApiEnvironment.PROD_AWS_URL) # e.g., PROD_AWS_URL, STAGING_AWS_URL, DEV2_AWS_URL, SOLUTIONS_GCP_URL
print(f"Logged in successfully as <{my_username}>.")

try:
    patients_ds   = session.dataset.get_dataset(PATIENTS_DATASET_UID)
    encounters_ds = session.dataset.get_dataset(ENCOUNTERS_DATASET_UID)
    procedures_ds = session.dataset.get_dataset(PROCEDURES_DATASET_UID)
    print("Datasets loaded successfully.")
except Exception as e:
    print(f"ERROR loading datasets: {e}")
    print("Confirm UIDs are correct and datasets are registered in Tutorial 1.")
    raise

## Step 2 — Row Counts

Start here. A successful count confirms the dataset is registered and the Rhino
client can access the CSV file.
**Expected:** 100 rows per dataset (including the intentional duplicate row).

In [ ]:
print(f"{'Dataset':<35} {'Row Count':>10}")
print("─" * 47)
for ds in [patients_ds, encounters_ds, procedures_ds]:
    try:
        result = ds.get_metric(Count(variable="patientID"))
        count = result.output["count"]
        print(f"{ds.name:<35} {count:>10,}")
    except Exception as e:
        print(f"{ds.name:<35} ERROR: {e}")

## Step 3 — Explore Patient Dataset

Profile `Gender`, `Race`, and `Ethnicity`. This reveals:
- Every **exact string value** that will need an OMOP vocabulary mapping (Tutorial 4)
- **Casing inconsistencies** to normalize in Tutorial 3
- **Null values** in required fields

Note: some rare values may be suppressed by the privacy engine (see note on
Differential Privacy below).

In [ ]:
import datetime

# Explore distributions of demographic variables in Patients dataset (Gender, Race & Ethnicity)
for col in ["Gender", "Race", "Ethnicity"]:
    try:
        result = patients_ds.get_metric(Count(
            variable=col,
            group_by={"groupings": [col]},
        ))
        print(f"patients → {col}")
        for group, value in result.output.items():
            print(f"  {group:<20} {value['count']}")
        print()
    except Exception as e:
        print(f"patients → {col}\n  ERROR: {e}")

# Explore distribution of YearOfBirth in Patients dataset and check for suspicious values
yob_mean = patients_ds.get_metric(Mean(variable="YearOfBirth"))
yob_std  = patients_ds.get_metric(StandardDeviation(variable="YearOfBirth"))
mean_val = yob_mean.output["mean"]
std_val  = yob_std.output["stddev"]
print(f"patients → YearOfBirth  mean={mean_val:.1f},  std={std_val:.1f}")
if mean_val > datetime.datetime.now().year or mean_val < 1900:
    print(f"  ⚠️  Mean year looks suspicious — check for invalid values")


## Step 4 — Explore Encounter Dataset

Check `TypeOfService` for casing inconsistencies and `DateOfService` for nulls.
`TypeOfService` values discovered here are exactly what we will map to OMOP
visit concepts in Tutorial 4 — every unique string must be covered.

In [ ]:
# Explore TypeOfService distribution in Encounters dataset and check for potential casing issues (e.g., "inpatient" vs "Inpatient")
print("encounters → TypeOfService")
try:
    result = encounters_ds.get_metric(Count(
        variable="TypeOfService",
        group_by={"groupings": ["TypeOfService"]},
    ))
    for value, counts in sorted(result.output.items(), key=lambda x: -x[1]["count"]):
        flag = " ← casing issue" if str(value) != str(value).title() else ""
        print(f"  {str(value):<20} {counts['count']:>6,}{flag}")
except Exception as e:
    print(f"  ERROR: {e}")

# Examine null rates — computed as (total rows) - (non-null count)
print("\nencounters — null rates:")
total = encounters_ds.get_metric(Count(variable="patientID")).output["count"]
for col in ["patientID", "visitID", "DateOfService"]:
    try:
        non_null = encounters_ds.get_metric(Count(variable=col)).output["count"]
        null_count = total - non_null
        null_pct = null_count / total * 100
        status = "✅" if null_count == 0 else "⚠️ "
        print(f"  {status} {col:<25} {null_pct:.1f}% null  ({null_count} rows)")
    except Exception as e:
        print(f"  ERROR checking {col}: {e}")

## Step 5 — Explore Procedure Dataset

Identify all CPT codes present in the data. Every code in this list must be
included in the CPT→OMOP semantic mapping in Tutorial 4.
Also flag the null `ProcedureCode` (row 92) which must be dropped in Tutorial 3.

In [ ]:
# Explore ProcedureCode distribution in Procedures dataset and check for nulls
print("procedures → ProcedureCode")
try:
    result = procedures_ds.get_metric(Count(
        variable="ProcedureCode",
        group_by={"groupings": ["ProcedureCode"]},
    ))
    print(f"\n  {'CPT Code':<12} {'Count':>6}")
    print("  " + "─" * 20)
    for code, counts in sorted(result.output.items(), key=lambda x: -x[1]["count"]):
        print(f"  {str(code):<12} {counts['count']:>6,}")
except Exception as e:
    print(f"  ERROR: {e}")

# Examine null rates — computed as (total rows) - (non-null count)
print("\nprocedures — null rates on key columns:")
total = procedures_ds.get_metric(Count(variable="patientID")).output["count"]
for col in ["patientID", "visitID", "ProcedureCode", "ProcedureDate"]:
    try:
        non_null = procedures_ds.get_metric(Count(variable=col)).output["count"]
        null_count = total - non_null
        null_pct = null_count / total * 100
        status = "✅" if null_count == 0 else "⚠️ "
        note = " ← rows with null code will be DROPPED in Tutorial 3" if col == "ProcedureCode" and null_count > 0 else ""
        print(f"  {status} {col:<25} {null_pct:.1f}% null  ({null_count} rows){note}")
    except Exception as e:
        print(f"  ERROR checking {col}: {e}")

## Step 6 — Understanding Differential Privacy & k-Anonymity

The Rhino FCP applies **privacy-preserving mechanisms** to all federated analytics
results. Understanding these helps interpret any surprising results.

### Differential Privacy
Differential privacy adds carefully calibrated mathematical noise to query
results, making it impossible to determine whether any specific individual's
data contributed to an aggregate result. The amount of noise is controlled by
the **epsilon** parameter set in your project's permission policy — lower epsilon
means more privacy (more noise), higher epsilon means less noise.

You may notice that histogram counts do not add up to exactly 100, or that
a mean value differs slightly from what you would compute directly. This is
differential privacy noise at work.

For more info, see https://docs.rhinofcp.com/security-and-data-protections/differential-privacy

### k-Anonymity
k-Anonymity suppresses query results that could be used to identify individuals
with small group sizes. If a histogram bin contains fewer than **k** records
(where k is set in the project permission policy), that bin is withheld entirely
rather than returned with a small count.

**Practical implications:**
- If you expected a histogram value but it is missing, it may have been suppressed
- A missing bin does not mean the data is wrong — it means that value is rare
- For Tutorial 4, ensure your semantic mapping covers all values *you know exist*
  even if some are suppressed in analytics results

**Where to check your permission policy:**
Dashboard → Projects → [Your Project] → Collaborators → Permissions Policy
This shows the differential privacy setting and k value parameters configured for your project.

## Step 7 — Federated Dataset (Multi-site Pattern)

A Federated Dataset groups multiple site datasets sharing the same schema.
In a multi-site project, this lets you run a single metric call that aggregates
across all sites. Here we demonstrate with one site — the code is identical
whether you have 1 or 10 sites.

In [ ]:
from rhino_health.lib.endpoints.federated_dataset.federated_dataset_dataclass import (
    FederatedDatasetCreateInput, DataSheet, PrivacySettings,
    AnalyticsVisibility, DifferentialPrivacy,
)

workgroup = session.project.get_collaborating_workgroups(PROJECT_UID)[0]
try:
    fed_dataset = session.federated_dataset.create_federated_dataset(FederatedDatasetCreateInput(
        name="Intro to Data Engineering - Federated Patients — All Sites",
        dataset_uids=[PATIENTS_DATASET_UID],       # add more site UIDs here for multi-site
        primary_workgroup_uid=workgroup.uid,
        datasheet=DataSheet(),
        analytics_visibility=AnalyticsVisibility.LIMITED,
        privacy_settings=PrivacySettings(
            k_anonymization_parameter=0,
            differential_privacy_setting=DifferentialPrivacy.NONE,
        ),
    ))
    print(f"Federated dataset <{fed_dataset.name}> created: {fed_dataset.uid}")
    print(f"\nFCP UI check: Dashboard → Federated Datasets (on the sidebar - click the layered cylinder icon)")
    print (f"Select the dataset that was just created.\nGo to the analytics tab to view the metrics computed across 'all sites' (which is just one site for now since we only included the original dataset).")
except Exception as e:
    print(f"Federated dataset creation failed: {e}")

# Aggregate metrics across all sites using the dataset UIDs directly
all_patient_uids = [PATIENTS_DATASET_UID]   # extend with more site UIDs for multi-site

fed_count = session.project.aggregate_dataset_metric(
    dataset_uids=all_patient_uids,
    metric_configuration=Count(variable="patientID"),
)
fed_yob = session.project.aggregate_dataset_metric(
    dataset_uids=all_patient_uids,
    metric_configuration=Mean(variable="YearOfBirth"),
)
print(f"\nAggregated across all sites:")
print(f"  Total rows:       {fed_count.output['count']:,}")
print(f"  Mean YearOfBirth: {fed_yob.output['mean']:.1f}")

## Step 8 — Federated SQL (External Database)

`run_sql_query` lets you run federated metrics directly against an **external SQL database** on the client node — without first importing the data as a dataset.

This is useful when your source data lives in PostgreSQL, MySQL, MSSQL, etc. and you want to run aggregate metrics (count, mean, etc.) on a query result before deciding whether to import.

> **This step is a reference template.** The tutorial data is CSV-based and already registered as datasets — use the federated metrics from steps 3–5 for those. Run this cell only if you have an external database to query.

**Supported database types:** `postgresql`, `mysql`, `mariadb`, `mssql`, `oracle`, `sqlite`, `iris`

In [ ]:
### UNCOMMENT IF YOU WANT TO RUN THIS CELL (otherwise, it's just for reference) — requires external DB connection details and will error if not configured

'''
from rhino_health.lib.endpoints.sql_query.sql_query_dataclass import (
    SQLQueryInput, ConnectionDetails, SQLServerTypes, SQLQueryImportInput
)

# Replace these placeholders with your actual database connection details
DB_SERVER_URL  = "mydbserver:5432"          # host:port
DB_SERVER_TYPE = SQLServerTypes.POSTGRESQL  # or MYSQL, MSSQL, SQLITE, etc.
DB_NAME        = "clinical_db"
DB_USER        = "db_user"
DB_PASSWORD    = getpass()

# Example SQL query to explore distribution of TypeOfService in encounters table
SQL_VISITS = """
    SELECT TypeOfService, COUNT(*) AS n_visits
    FROM encounters
    GROUP BY TypeOfService
    ORDER BY n_visits DESC
"""
try:
    result = session.sql_query.run_sql_query(SQLQueryInput(
        session=session,
        project_uid=PROJECT_UID,
        workgroup_uid=workgroup.uid,
        sql_query=SQL_VISITS,
        connection_details=ConnectionDetails(
            server_url=DB_SERVER_URL,
            server_type=DB_SERVER_TYPE,
            db_name=DB_NAME,
            server_user=DB_USER,
            password=DB_PASSWORD,
        ),
        metric_definitions=[Count(variable="TypeOfService")],
        timeout_seconds=300,
    ))
    print("SQL query result:")
    print(result.results)
except Exception as e:
    print(f"SQL query skipped (no external DB configured): {e}")

# To import the query result as a registered FCP dataset instead, use:
result = session.sql_query.import_dataset_from_sql_query(SQLQueryImportInput(
    session=session, 
    project_uid=PROJECT_UID,
    workgroup_uid=workgroup.uid,
    sql_query="SELECT * FROM encounters",
    connection_details=ConnectionDetails(
          server_url=DB_SERVER_URL,
          server_type=DB_SERVER_TYPE,
          db_name=DB_NAME,
          server_user=DB_USER,
          password=DB_PASSWORD,
    ),
    dataset_name="Encounters — Imported from DB",
    is_data_deidentified=True,
    timeout_seconds=600,
))
result.wait_for_completion()
'''

## Step 9 — Interactive Containers (for deeper exploration)

All metrics above return aggregates without exposing raw data. When you need
to interactively explore the actual data on the client (e.g., to understand
a specific anomaly, or to prototype a cleaning script), use an **Interactive
Container**.

An Interactive Container (IC) is essentially a pre-packaged Docker container that includes specific dependencies, such as Jupyter Notebook. It launches a live session that runs directly
on the Rhino client, with your dataset mounted at `/input/dataset.csv`. If you have multiple inputs, they will be mounted at `/input/0/dataset.csv`, ``/input/1/dataset.csv`, and so on...

You access the IC through a browser, and no data leaves the client network.

**The IC is launched from the FCP Dashboard / UI, NOT this notebook:**

1. Dashboard → Projects → [Your Project] → Code → New Code Object
2. Select **Interactive Container** as the type
3. Choose the appropriate container (e.g., interactive-jupyter-notebook)
4. Click Run & Select Dataset Inputs
5. Go to the **Code Runs** tab - when status shows **Running**, click the Arrow Button next to IC (**Type**) to open the interactive container in a new browser tab
6. Click **Terminate** when done

Use interactive containers to:
- Browse specific rows that have quality issues
- Prototype and test cleaning code before formalizing it as a Code Object
- Debug a failing Code Object run by inspecting what the input data looks like

> **Note:** If you do not have the ability to view secure data for a dataset, you will not be able to select it as a data input!!

## Step 10 - FCP UI — What to Check After Running This Notebook

1. **Dataset Analytics view** → Dashboard → Projects → [Your Project] → Datasets → click any dataset → **Analytics tab**
   - Provides an automatic column profile (null rates, distributions)
   - Compare with your notebook output to confirm consistency

2. **Federated Dataset** → Dashboard → Federated Datasets
   - Your new federated dataset appears
   - Click it to explore Datasheet, Data Schema, Access, and Analytics

3. **Permission Policy** → Dashboard → Projects → [Your Project] → Collaborators → Permissions Policy
   - Check your project's differential privacy and k-anonymity settings
   - This can explain any suppressed histogram bins or slightly noisy aggregate values

## Step 11 — Discovery Summary

Document findings before moving to Tutorial 3. This list drives every decision
in data preparation and harmonization.

In [ ]:
print("""
┌────────────────────────────────────────────────────────────────┐
│  Data Discovery Summary — fill in before continuing            │
├────────────────────────────────────────────────────────────────┤
│                                                                │
│  Issues to fix in Tutorial 3 (Data Preparation):               │
│    Gender: casing variants found (female, FEMALE, male, MALE)  │
│    Gender: null values present → will be flagged/dropped       │
│    Gender: invalid value "Alien" → will be flagged/dropped     │
│    YearOfBirth: value 10000 found → will be dropped            │
│    TypeOfService: casing variants (outpatient, INPATIENT, etc) │
│    DateOfService: null in 1 row → row will be dropped          │
│    ProcedureCode: null in 1 row → row will be dropped          │
│    Duplicates: 1 duplicate row in each dataset → will be dedup │
│                                                                │
│  CPT codes to map in Tutorial 4 (all must be in mapping):      │
│    45378, 44950, 99203, 99212, 99213, 99285                    │
│                                                                │
│  Gender values to map (exact strings after cleaning):          │
│    Male, Female, Other                                         │
│                                                                │
│  TypeOfService values to map (after cleaning):                 │
│    Outpatient, Inpatient, Emergency                            │
│                                                                │
└────────────────────────────────────────────────────────────────┘
""")

print("\nNote that the above issues are based on the sample data in this tutorial and may differ from what you see in your own datasets.")
print("Also note that due to differential privacy noise & k-anonymity, the exact counts and distributions you see may differ from the raw data — this is expected behavior to protect patient privacy.")

print("\nContinue to: Tutorial 3 - Data_Preparation")